#3 Homework about Classification

#1. Q1: Mode for industry

In [ ]:
#Data preparation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

!wget $data -O course_lead_scoring_2026.csv



--2026-10-12 19:08:36--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.007s  

2026-10-12 19:08:36 (37.8 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [2]:
df= pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
#Check if the missing values are presented in the features
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
#list out datatype for each feature
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [9]:
categorical_features= df.select_dtypes(include=['str']).columns
numerical_features= df.select_dtypes(include=['int64', 'float64']).columns
categorical_features, numerical_features

(Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='str'),
 Index(['annual_income', 'number_of_courses_viewed', 'interaction_count',
        'lead_score', 'converted'],
       dtype='str'))

In [10]:
for cf in categorical_features:
    if df[cf].isnull().sum() > 0:
        df[cf] = df[cf].fillna('NA')

In [11]:
for nf in numerical_features:
    if df[nf].isnull().sum()>0:
        df[nf] = df[nf].fillna(0.0)

In [13]:
df.isnull().sum()
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [18]:
industry_mode= df.industry.mode()[0]

print(f"Most frequent industry: {industry_mode}")
df.industry.value_counts()

Most frequent industry: technology


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

#2 Q2: Biggest correlation between numerical features

In [19]:
#list out correlation between numerical features
correlation_matrix = df[numerical_features].corr()
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


#3 Q3: Biggest MI

In [30]:
#Data validation framework
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [25]:
len(df_train), len(df_val), len(df_test)
#Reset the index for all three datasets
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [27]:
#Isolate the target variable from the features
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [28]:
#Delete the target variable from the features
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [34]:
mutual_info_score(df_train['industry'], y_train)
for cf in categorical_features:
    mi = round(mutual_info_score(df_train[cf], y_train), 2)
    print(f"Mutual information between {cf} and converted: {mi}")

Mutual information between lead_source and converted: 0.03
Mutual information between industry and converted: 0.0
Mutual information between employment_status and converted: 0.02
Mutual information between location and converted: 0.0


#4 Q4: Accuraty

In [53]:
features = list(categorical_features) + list(numerical_features)
#remove converted from features
features = [f for f in features if f != "converted"]
features
def train_and_score(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feats].to_dict(orient='records'))
    X_val = dv.transform(df_val[feats].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return accuracy_score(y_val, model.predict(X_val))

accuracy = train_and_score(features)
accuracy, round(accuracy, 2)

(0.645, 0.65)

#5 Q5: Feature selection

In [55]:
differences = {}
for f in features:
    acc_without = train_and_score([x for x in features if x != f])
    differences[f] = accuracy - acc_without
    print(f'{f:26s} acc_without={acc_without:.3f}  diff={accuracy - acc_without:+.3f}')

{f: round(differences[f], 3) for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']}

lead_source                acc_without=0.642  diff=+0.003
industry                   acc_without=0.645  diff=+0.000
employment_status          acc_without=0.645  diff=+0.000
location                   acc_without=0.645  diff=+0.000
annual_income              acc_without=0.724  diff=-0.079
number_of_courses_viewed   acc_without=0.643  diff=+0.002
interaction_count          acc_without=0.601  diff=+0.044
lead_score                 acc_without=0.644  diff=+0.001


{'lead_source': 0.003,
 'number_of_courses_viewed': 0.002,
 'interaction_count': 0.044}

#6 Q6 : Parameter tuning 

In [56]:


for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(features, C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
